# Konvolúciós neurális hálózatok

A gyakorlaton KRESZ táblákat fogunk osztályozni konvolúciós neurális hálózatokkal. Ehhez először letöltjük, majd beolvassuk az adatokat:

In [ ]:
!wget 'https://github.com/bolgarbe/gt-data/raw/master/kresz.npz'

--2024-03-12 11:17:42--  https://github.com/bolgarbe/gt-data/raw/master/kresz.npz
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/bolgarbe/gt-data/master/kresz.npz [following]
--2024-03-12 11:17:43--  https://raw.githubusercontent.com/bolgarbe/gt-data/master/kresz.npz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 69108046 (66M) [application/octet-stream]
Saving to: ‘kresz.npz’

kresz.npz           100%[===================>]  65.91M   245MB/s    in 0.3s    

2024-03-12 11:17:43 (245 MB/s) - ‘kresz.npz’ saved [69108046/69108046]



In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import torch

In [ ]:
data = np.load('kresz.npz')
x_train = data['x_train']
x_test  = data['x_test']
y_train = data['y_train']
y_test  = data['y_test']

Az `x_train` változó a tanítóhalmaz képeit, az `y_train` változó a hozzájuk tartozó címkéket tárolja. Próbaképpen vizsgáljuk meg, hogy mit is tartalmaznak ezek a változók:

In [ ]:
# Kommenteld ki, futtasd le!
#x_train
#y_train

Jobban látszik, ha kiírjuk a dimenziókat:

In [ ]:
print(x_train.shape)
print(y_train.shape)

(4575, 32, 32, 3)
(4575,)


Tehát az `x_train` változó $4575$ darab $32\times 32$ RGB képet tárol (ez $4575\times 32\times 32\times 3$ darab, $0$ és $1$ közötti valós szám); hasonlóképpen, az `y_train` változóban 4575 címke szerepel ($0$ és $61$ közötti egész számok).

**1. feladat.** Vizsgáld meg, hogy mely típusú KRESZ-táblából van a legtöbb, ill. legkevesebb a tanító adatbázisban, rajzolj ki egy-egy példát!

A `PyTorch` könyvtár az adatokat saját adattípusba (`torch.Tensor`) csomagolva várja. Képek esetén az adatszerkezet némiképp eltér az eddigitől:

(`mintaszám` $\times$ `sor` $\times$ `oszlop` $\times$ `szín`) helyett (`mintaszám` $\times$ `szín` $\times$ `sor` $\times$ `oszlop`)

Ezért a tenzorrá alakításnál át kell variálnunk a sorrendet:

In [ ]:
x_train_tensor = torch.tensor(x_train).permute(0,3,1,2)
y_train_tensor = torch.tensor(y_train)

**2. feladat.** Hozz létre egy `DataLoader` objektumot, ami az adat minibatch-ekre bontását fogja végezni! Állíts be megfelelő batch méretet, illetve gondoskodj róla, hogy az adatok véletlenszerű sorrendben érkezzenek.

Dokumentáció:
https://pytorch.org/docs/stable/data.html

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(x_train_tensor,y_train_tensor)
loader = DataLoader(dataset,batch_size=256)

**3. feladat.** Hozz létre az előadáson bemutatotthoz hasonló konvolúciós neurális architektúrát a képek osztályozására! (Első körben csak futtasd le ezt, később kell majd finomhangolni a megfelelő prediktív teljesítmény elérése érdekében.)

Dokumentáció:

https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html

https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity

https://pytorch.org/docs/stable/generated/torch.nn.Linear.html


In [ ]:
# Egyszerű háló egyetlen konvolúciós réteggel és ReLU aktivációval
net = torch.nn.Sequential(                          # [Bemenet:                               -> (batch_size x 3 x 32 x 32)]
  torch.nn.Conv2d(3,32,(4,4),stride=2,padding=1),   # Konvolúciós réteg                       -> (batch_size x 32 x 16 x 16)
  torch.nn.ReLU(),                                  # ReLU aktiváció                          -> (batch_size x 32 x 16 x 16)
  torch.nn.Flatten(),                               # Besimítás 2D képekből 1D vektorokká     -> (batch_size x 8192)
  torch.nn.Linear(8192,62)                          # Teljesen összekötött réteg a 62 címkére -> (batch_size x 62)
)

A modell tanításához választunk egy alkalmas optimalizálót (pl. ADAM) és megadjuk a veszteségfüggvényt (osztályozást végzünk, tehát keresztentrópiát kell használnunk).

**4. feladat.** Válassz egy megfelelő learning rate-et és epoch számot, majd tanítsd a modellt az alábbi training loop futtatásával!

Dokumentáció: https://pytorch.org/docs/stable/generated/torch.optim.Adam.html

In [ ]:
opt = torch.optim.Adam(net.parameters())
ce  = torch.nn.CrossEntropyLoss()

# alap PyTorch training loop
num_epoch = 10
net.train()                        # training "üzemmód"
for epoch in range(num_epoch):     # végigmegyünk az adatokon num_epoch-szor
  for x,y in loader:             # végigiterálunk a minibatch-eken
    opt.zero_grad()            # gradiens kinullázása (NE FELEJTSD EL!)
    out = net(x)               # hálózat kimenete
    loss = ce(out,y)           # loss kiszámítása
    loss.backward()            # gradiens kiszámítása (backpropagation)
    opt.step()                 # modellparaméterek frissítése

A tanítás után nincs más dolgunk, mint ráadni a teszt bemeneteket a hálóra.

**5. feladat.** Alakítsd át a teszt adatokat a tanítókhoz hasonlóan, majd futtasd le a modellt és értékeld ki a prediktív teljesítményt (pontosság)!

**A pontosság haladja meg a 97%-ot!** (Addig variáld az architektúrát és a hiperparamétereket, amíg ez be nem következik.)

In [ ]:
#with torch.no_grad():
  # out = net(...)
  # ...

**6. feladat.** Mutasd be az osztályozót néhány véletlenszerűen választott képen a teszt halmazból! (pl. kép és a jósolt címke kirajzolása).

**7. feladat (szorgalmi).** Vizsgáld meg a prediktív teljesítményt egy gyakori és egy ritka osztályon a bináris klasszifikációnál tanult metrikák alapján (ROC és PR görbe alatti terület).